# hoodini CLI Workflow Demo

This notebook runs all main parts of the `hoodini` CLI workflow, step by step, as implemented in the command line tool. Each cell corresponds to a logical section of the pipeline.

In [2]:
# Patch asyncio for Jupyter compatibility
import nest_asyncio
nest_asyncio.apply()
print("Asyncio patched for Jupyter compatibility.")

Asyncio patched for Jupyter compatibility.


## 1. Import Required Libraries
Import all necessary libraries and modules used in cli.py, including any custom modules.

In [3]:
import os
import sys
import logging
import pandas as pd
import rich_click as click
import tomli
from pathlib import Path
from hoodini.config import load_default_config
from hoodini.utils.cli_helpers import MutuallyExclusiveOption
from hoodini.utils.core import validate_input_file, validate_domains
from hoodini.utils.logging_utils import console, logger, stage_header, stage_done, run_with_spinner

## 2. Parse Command-Line Arguments
Simulate or manually define the command-line arguments that would be passed to cli.py, and parse them as needed.

In [4]:
# Initialize console and logger (these are already available as imports)
# Test logging
logger.info("Hoodini workflow notebook initialized")

[19:21:10] INFO     Hoodini workflow notebook initialized                                           ]8;id=558049;file:///tmp/ipykernel_1868100/1581396180.py\1581396180.py]8;;\:]8;id=425716;file:///tmp/ipykernel_1868100/1581396180.py#3\3]8;;\

## 3. Initialize Main Workflow Components
Set up any objects, configurations, or initial states required for the workflow.(THIS IS OPTIONAL)

In [5]:
class ConfigObj(dict):
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__

# Initialize configuration and logging
config = load_default_config()

# Display loaded config for reference
console.print('Loaded configuration:', config)

config = ConfigObj(config)

Loaded configuration:
{
    'output': 'results',
    'num_threads': 10,
    'max_concurrent_downloads': 8,
    'apikey': '',
    'mod': 'win_nts',
    'wn': 20000,
    'height_factor': 20,
    'ngenes': 10,
    'minwin': 2000,
    'minwin_type': 'both',
    'tree_mode': 'fast_nj',
    'tree_file': 'target_prots.nwk',
    'nj_algorithm': 'nj',
    'aai_mode': 'wgrr',
    'aai-subset-mode': 'target_region',
    'ani_mode': 'fastani',
    'cand_mode': 'best_id',
    'clust_method': 'diamond_deepclust',
    'padloc': False,
    'deffinder': False,
    'ncrna': False,
    'cctyper': False,
    'domains': '',
    'domains_metadata': '',
    'min_prevalence': 0.0,
    'genomad': False,
    'antidefense': False,
    'phrogs': False,
    'sorfs': False,
    'prot_links': False,
    'nt_links': False,
    'nt_aln_mode': 'blastn',
    'assembly_folder': '',
    'assembly_db': '/env/db/hoodini',
    'img_db': '',
    'img_nuc': '',
    'blast': '',
    'img': '/mnt/fastdb/imgpr_imgvr/data_structure/',
    'img_metadata': '/mnt/fastdb/imgpr_imgvr/merged_metadata/imgvr_pr_taxids.txt',
    'keep': False,
    'force': False
}

## 4. Execute Core Workflow Logic
Run the main logic of the workflow, step by step, as implemented in cli.py.

## Initialize inputs

In [6]:

# Run main workflow logic (simplified for notebook)
# 1. Parse input files
console.print(f'Parsing input file: {config.input_file}')


from hoodini.initialize import initialize_inputs

# Initialize the records DataFrame
records = initialize_inputs(
    input_path="cas9.txt",
    output=config.output,
)

Parsing input file: None

📁 Assembly DB found: /home/klaupaucius/software/hoodini/hoodini/data/assembly_summary.parquet (updated: 
2025-08-29)

✅ Using existing database (less than 1 month old).

✔️  Created folder results.

In [7]:
records

,og_index,protein_id,nucleotide_id,uniprot_id,img,failed,gff_path,faa_path,fna_path,strand,start,end,gbf_path,taxid,assembly_id,input_type,premade
0,0,WP_217844005.1,None,None,None,None,None,None,None,None,None,None,None,None,None,protein,None
1,1,WP_347132630.1,None,None,None,None,None,None,None,None,None,None,None,None,None,protein,None
2,2,WP_239738697.1,None,None,None,None,None,None,None,None,None,None,None,None,None,protein,None
3,3,WP_198985183.1,None,None,None,None,None,None,None,None,None,None,None,None,None,protein,None
4,4,WP_262856654.1,None,None,None,None,None,None,None,None,None,None,None,None,None,protein,None
5,5,WP_107596301.1,None,None,None,None,None,None,None,None,None,None,None,None,None,protein,None
6,6,WP_262588072.1,None,None,None,None,None,None,None,None,None,None,None,None,None,protein,None
7,7,WP_107544007.1,None,None,None,None,None,None,None,None,None,None,None,None,None,protein,None
8,8,WP_232167892.1,None,None,None,None,None,None,None,None,None,None,None,None,None,protein,None
9,9,OJH01299.1,None,None,None,None,None,None,None,None,None,None,None,None,None,protein,None


In [8]:
from hoodini.parse_ipg import run_ipg

records = run_ipg(
    records_df=records,  # or provide a DataFrame if you already have one
    cand_mode="best_id",)

console.print("IPG step complete. Records loaded.")

Output()

IPG step complete. Records loaded.

In [9]:
from hoodini.parse_assemblies import run_assembly_parser
result = run_assembly_parser(
    records_df=records,
    output_dir=config.output,
    assembly_folder=config.assembly_folder,
    ncrna=config.ncrna,
    cctyper=config.cctyper,
    genomad=config.genomad,
    blast=config.blast,
    apikey=config.apikey,
    max_concurrent_downloads=config.max_concurrent_downloads,
    img=config.img,
    num_threads=config.num_threads,
    mod=config.mod,
    wn=config.wn,
    sorfs=config.sorfs,
    minwin=config.minwin,
    minwin_type=config.minwin_type,
)

✔️  Downloaded or located assemblies (folder: results/assembly_folder)

3 contigs below min window size: [1, 2, 7]

✔️  Extracted neighborhoods and wrote output files.

In [10]:
records = result["records"]
all_gff = result["all_gff"]
all_prots = result["all_prots"]
all_neigh = result["all_neigh"]
valid_uids = result["valid_uids"]

In [11]:
all_prots

,protein_id,id,sequence,product,target_prot,target_nuc,unique_id
0,WP_058590966.1,WP_058590966.1,MKILDVISQIEQEILVSNNNYKINIDKECTDITTMYQNIKTSSVFI...,Mur ligase family protein,WP_217844005.1,NZ_JALKNF010000002.1,0
1,WP_084756317.1,WP_084756317.1,MNISQKVLAMSLSVSLLGAGTSVSQIAQAHDASHESVTKTDSKAVE...,copper amine oxidase,WP_217844005.1,NZ_JALKNF010000002.1,0
2,WP_058590968.1,WP_058590968.1,MECEKLYDYFNDQLSKDEKQKFEEHLKTCKNCQEELDELNILNESL...,anti-sigma factor,WP_217844005.1,NZ_JALKNF010000002.1,0
3,WP_025905379.1,WP_025905379.1,MRESDIELYDKIVSKDTDSLETLYDRYESLLYSFIYKITQDKQSSE...,RNA polymerase sigma factor,WP_217844005.1,NZ_JALKNF010000002.1,0
4,WP_058590969.1,WP_058590969.1,MRKMITVLMTVLILSGCHNQMNHVDKKTSEEKNVQTVESKAFETYQ...,class F sortase,WP_217844005.1,NZ_JALKNF010000002.1,0
...,...,...,...,...,...,...,...
246,OJH01253.1,OJH01253.1,MPSANQKRMIKHIHETVFILLHDYHFDEITVQKICDIAEINRSTFY...,TetR family transcriptional regulator,OJH01299.1,MPNR01000008.1,9
247,OJH01254.1,OJH01254.1,MRKQQDNHSAYVFIKRLIKQFGKPQKVITDQAPSTKVAMAKVIKAF...,IS6 family transposase,OJH01299.1,MPNR01000008.1,9
248,OJH01255.1,OJH01255.1,MKIKKLLKTLLIILLCFVLSVVVQNISMLWHIASIWSIEYILHALT...,CAAX protease,OJH01299.1,MPNR01000008.1,9
249,OJH01301.1,OJH01301.1,MMISKQIKGLRKQHNYTQEELAEKLNTSRQTISKWEKGISEPDLNM...,transcriptional regulator,OJH01299.1,MPNR01000008.1,9


In [12]:
from hoodini.cluster_proteins import cluster_proteins  # assuming it's saved here

cluster = cluster_proteins(
    all_prots,
    output_dir=config.output,
    clust_method=config.clust_method,
    sorfs=config.sorfs
)

✔️       Protein clustering complete

In [13]:
cluster

,protein_id,id,sequence,product,target_prot,target_nuc,unique_id,fam_cluster
0,WP_058590966.1,WP_058590966.1,MKILDVISQIEQEILVSNNNYKINIDKECTDITTMYQNIKTSSVFI...,Mur ligase family protein,WP_217844005.1,NZ_JALKNF010000002.1,0,NaN
1,WP_084756317.1,WP_084756317.1,MNISQKVLAMSLSVSLLGAGTSVSQIAQAHDASHESVTKTDSKAVE...,copper amine oxidase,WP_217844005.1,NZ_JALKNF010000002.1,0,NaN
2,WP_058590968.1,WP_058590968.1,MECEKLYDYFNDQLSKDEKQKFEEHLKTCKNCQEELDELNILNESL...,anti-sigma factor,WP_217844005.1,NZ_JALKNF010000002.1,0,NaN
3,WP_025905379.1,WP_025905379.1,MRESDIELYDKIVSKDTDSLETLYDRYESLLYSFIYKITQDKQSSE...,RNA polymerase sigma factor,WP_217844005.1,NZ_JALKNF010000002.1,0,NaN
4,WP_058590969.1,WP_058590969.1,MRKMITVLMTVLILSGCHNQMNHVDKKTSEEKNVQTVESKAFETYQ...,class F sortase,WP_217844005.1,NZ_JALKNF010000002.1,0,NaN
...,...,...,...,...,...,...,...,...
246,OJH01253.1,OJH01253.1,MPSANQKRMIKHIHETVFILLHDYHFDEITVQKICDIAEINRSTFY...,TetR family transcriptional regulator,OJH01299.1,MPNR01000008.1,9,NaN
247,OJH01254.1,OJH01254.1,MRKQQDNHSAYVFIKRLIKQFGKPQKVITDQAPSTKVAMAKVIKAF...,IS6 family transposase,OJH01299.1,MPNR01000008.1,9,NaN
248,OJH01255.1,OJH01255.1,MKIKKLLKTLLIILLCFVLSVVVQNISMLWHIASIWSIEYILHALT...,CAAX protease,OJH01299.1,MPNR01000008.1,9,NaN
249,OJH01301.1,OJH01301.1,MMISKQIKGLRKQHNYTQEELAEKLNTSRQTISKWEKGISEPDLNM...,transcriptional regulator,OJH01299.1,MPNR01000008.1,9,NaN


In [ ]:
from hoodini.taxonomy import parse_taxonomy_and_build_tree

tree_str, den_data = parse_taxonomy_and_build_tree(
    records=records,
    all_gff=all_gff,
    all_neigh=all_neigh,
    all_prots=all_prots,
    output_dir=config.output,
    tree_mode=config.tree_mode,
    tree_file=config.tree_file,
    num_threads=config.num_threads,
    aai_mode=config.aai_mode,
    ani_mode=config.ani_mode,
    aai_subset_mode=config.aai_subset_mode,
    nj_algorithm=config.nj_algorithm,
)

In [ ]:
from hoodini.protein_links import run_protein_links
pairwise_aa = run_protein_links(
    output_dir=config.output,
    all_prots=all_prots,
    threads=config.num_threads,
    evalue=1e-5
)

In [ ]:
from hoodini.pairwise_nt import run_pairwise_nt

pairwise_ani, nt_links = run_pairwise_nt(
    all_neigh=all_neigh,
    all_gff=all_gff,
    output_dir=config.output,
    nt_aln_mode="blastn",
    ani_mode="blastn",
    nt_links=True,
    threads=config.num_threads,
)

In [14]:
from hoodini.extra_tools.domain import run_domain
# config.domains is already a validated list of database names
config.domains = ["pfam","phrog"]
domains_data = run_domain(all_prots, config.output, config.domains, config.num_threads)

🔍      Annotating domains for pfam with HMMER...

🔍      Annotating domains for phrog with HMMER...

✔️       Domain annotation complete: 595 matches from 2 databases (deduplicated from 1371).

In [15]:
domains_data

,protein_id,domain_id,bit_score,alignment_length,e_value,start,end,cov,database,bit_score*alignment_length,domain_id_clean,ID,Function_pfam,Clan_pfam,Gene_pfam,L1_phrog,Function_phrog,EC_phrog
894,OJH01224.1,PF03773,206.836273,286,2.069298e-65,5,291,0.972789,pfam,59155.174133,PF03773,PF03773,Predicted permease,CL0182,ArsP_1,NaN,NaN,NaN
891,OJH01225.1,PF01022,50.070873,47,4.100367e-17,28,75,0.443396,pfam,2353.331043,PF01022,PF01022,"Bacterial regulatory protein, arsR family",CL0123,HTH_5,NaN,NaN,NaN
1368,OJH01225.1,phrog_678,23.863550,90,2.497653e-09,5,95,0.849057,phrog,2147.719517,phrog_678,phrog_678,NaN,NaN,NaN,"DNA, RNA and nucleotide metabolism",replication initiation O-like,NaN
890,OJH01227.1,PF06953,132.057785,114,7.266120e-43,1,115,0.991304,pfam,15054.587494,PF06953,PF06953,Arsenical resistance operon protein ArsD,CL0172,ArsD,NaN,NaN,NaN
795,OJH01228.1,PF02374,194.761871,280,1.084038e-61,20,300,0.486111,pfam,54533.323975,PF02374,PF02374,Anion-transporting ATPase,CL0023,ArsA_ATPase,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
901,WP_347132630.1,phrog_8660,-4.053548,17,6.782985e+00,47,64,0.018162,phrog,-68.910314,phrog_8660,phrog_8660,NaN,NaN,NaN,"DNA, RNA and nucleotide metabolism",HNH endonuclease,NaN
899,WP_347132630.1,phrog_1035,-4.092455,0,8.005324e+00,133,133,0.000000,phrog,-0.000000,phrog_1035,phrog_1035,NaN,NaN,NaN,"DNA, RNA and nucleotide metabolism",endonuclease VII,NaN
902,WP_347132630.1,phrog_1035,-6.780001,28,1.000000e+01,741,769,0.029915,phrog,-189.840033,phrog_1035,phrog_1035,NaN,NaN,NaN,"DNA, RNA and nucleotide metabolism",endonuclease VII,NaN
1,WP_398577366.1,PF13303,328.495605,333,1.784749e-101,21,354,0.932773,pfam,109389.036621,PF13303,PF13303,"Phosphotransferase system, EIIC",CL0493,PTS_EIIC_2,NaN,NaN,NaN


In [ ]:
from hoodini.extra_tools.padloc import run_padloc
padloc_df = run_padloc(all_gff, all_prots, config.output, config.num_threads)
all_prots = all_prots.merge(padloc_df, on="id", how="left")
from hoodini.extra_tools.defensefinder import run_defensefinder
deffinder_df = run_defensefinder(all_gff, all_prots, config.output)
if not deffinder_df.empty:
    all_prots = all_prots.merge(deffinder_df, on="id", how="left")

In [ ]:

from hoodini.extra_tools.blast import run_blast
blast_data = run_blast(all_neigh, config.output, "query.fa", config.num_threads, valid_uids)
#convert to GFF-like format in which start = qstart, end=qend, ID=qseqid, feature type should be "region"
if not blast_data.empty:
    gff_df = pd.DataFrame({
        "seqid": blast_data["seqid"],
        "source": "hoodini",
        "type": "region",
        "start": blast_data["start"],
        "end": blast_data["end"],
        "score": ".",
        "strand": "+",
        "phase": ".",
        "attributes": "ID=" + blast_data["nc_feature"] + ";"
    })
    all_gff = pd.concat([all_gff, gff_df], ignore_index=True)


from hoodini.extra_tools.cctyper import run_cctyper
cctyper_df, crispr_df = run_cctyper(all_gff, all_prots, all_neigh, config.output, config.num_threads, valid_uids)
#if cctyper_df is not empty:
if not cctyper_df.empty:
    all_prots = all_prots.merge(cctyper_df, on="id", how="left")
if not crispr_df.empty:
    gff_df = pd.DataFrame({
        "seqid": crispr_df["Contig"],
        "source": "hoodini",
        "type": "region",
        "start": crispr_df["start"],
        "end": crispr_df["end"],
        "score": ".",
        "strand": ".",
        "phase": ".",
        "attributes": "ID=" + crispr_df["nc_feature"] + ";"
    })
    #append to all_gff
    all_gff = pd.concat([all_gff, gff_df], ignore_index=True)

from hoodini.extra_tools.genomad import run_genomad
genomad_df = run_genomad(all_neigh, config.output, config.num_threads, valid_uids)
if not genomad_df.empty:
    gff_df = pd.DataFrame({
        "seqid": genomad_df["seqid"],
        "source": "hoodini",
        "type": "region",
        "start": genomad_df[["start", "end"]].min(axis=1),
        "end": genomad_df[["start", "end"]].max(axis=1),
        "score": ".",
        "strand": ".Z",
        "phase": ".",
        "attributes": "ID=" + genomad_df["mge_type"] + ";"
    })
    all_gff = pd.concat([all_gff, gff_df], ignore_index=True)


from hoodini.extra_tools.ncrna import run_ncrna
ncrna_data = run_ncrna(all_neigh, den_data, config.output, config.num_threads, valid_uids)
if not ncrna_data.empty:
    gff_df = pd.DataFrame({
        "seqid": ncrna_data["nucid"],
        "source": "hoodini",
        "type": "ncRNA",
        "start": ncrna_data[["start", "end"]].min(axis=1),
        "end": ncrna_data[["start", "end"]].max(axis=1),
        "score": ".",
        "strand": ncrna_data["strand_ncrna"],
        "phase": ".",
        "attributes": "ID=" + ncrna_data["nc_feature"] + ";"
    })
    all_gff = pd.concat([all_gff, gff_df], ignore_index=True)

In [ ]:
# Use the configured output directory directly
outdir = Path(config.output+ "/hoodini-viz")
outdir.mkdir(parents=True, exist_ok=True)

# GFF
all_gff.to_csv(outdir / "defaultGFF.gff", index=False, sep="\t", header=False)

# Baselines
all_neigh = all_neigh.rename(columns={
    "unique_id": "hood_id",
    "start_win": "start",
    "end_win": "end",
    "target_prot": "align_gene",
})
all_neigh[["hood_id", "seqid", "start", "end", "align_gene"]].to_csv(
    outdir / "defaultBaselines.txt", sep="\t", index=False
)

# Protein metadata
all_prots = all_prots.rename(columns={"protein_id": "gene_id", "fam_cluster": "cluster"})
all_prots["product"] = all_prots["product"].str.replace("\n", " ")
all_prots.to_csv(outdir / "defaultProteinMetadata.txt", sep="\t", index=False)

# Tree metadata
den_data = den_data.rename(columns={"unique_id": "leaf_id"})
den_data["leaf_id"] = den_data["leaf_id"].astype(str).str.extract(r"(\d+)").astype(int)
den_data.to_csv(outdir / "defaultTreeMetadata.txt", sep="\t", index=False)

# Newick tree
with open(outdir / "defaultNewick.txt", "w", encoding="utf-8") as f:
    f.write(tree_str)
    
# Domain metadata
if config.domains:
    with open(outdir / "defaultDomains.txt", "w", encoding="utf-8") as f:
        domains_data[["protein_id","domain_id","start","end","database"]].to_csv(f, sep="\t", index=False, header=False)
    
# Nucleotide link
# write nucleotide links; if nt_links not produced, write placeholder header-only file
if nt_links:
    nt_links[["query","query_start","query_end","ref","ref_start","ref_end","ani"]].to_csv(outdir / "defaultNucleotideLinks.txt", sep="\t", index=False,header=False)
else:
    pd.DataFrame(columns=["query","query_start","query_end","ref","ref_start","ref_end","ani"]).to_csv(outdir / "defaultNucleotideLinks.txt", sep="\t", index=False,header=False)

# Protein links
# pairwise_aa is only created when protein comparisons were run; if missing,
# write an empty placeholder file with the expected columns so downstream
# consumers always find the file.
if pairwise_aa:
    pairwise_aa[["qseqid","sseqid","pident"]].to_csv(outdir / "defaultProteinLinks.txt", sep="\t", index=False,header=False)
else:
    pd.DataFrame(columns=["qseqid","sseqid","pident"]).to_csv(outdir / "defaultProteinLinks.txt", sep="\t", index=False,header=False)

